# Phase 6: 構造化RAGシステム実験

**目的**: 座標計算・集計・比較機能を統合した構造化RAGシステムの評価

**期待効果**:
- spatial_comparison: -8.9pt → +5pt以上
- advanced_comparison: -0.8pt → +10pt以上

**作成日**: 2026-01-22

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers
!pip install -q tqdm pandas matplotlib japanize-matplotlib

print("パッケージインストール完了")

In [ ]:
# 1.2 GPU確認
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_memory:.1f} GB")
else:
    print("GPUが利用できません。ランタイムを変更してください。")

In [ ]:
# 1.3 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.4 パス設定
import os
import sys

BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results"
SRC_DIR = f"{BASE_DIR}/src"

# ディレクトリ作成
for dir_path in [DATA_DIR, RESULTS_DIR, SRC_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# srcをパスに追加
sys.path.insert(0, BASE_DIR)

print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.5 モデル設定
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

print(f"LLM: {LLM_MODEL}")
print(f"Embedding: {EMBEDDING_MODEL}")

## Section 2: データ読み込み

In [ ]:
# 2.1 POIデータ読み込み
import json

poi_path = f"{DATA_DIR}/poi_documents.json"

with open(poi_path, "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

print(f"POIデータ読み込み完了: {len(poi_documents)}件")

In [ ]:
# 2.2 POIデータの形式を統一
all_pois = []
for doc in poi_documents:
    if "metadata" in doc:
        poi = doc["metadata"].copy()
        poi["content"] = doc.get("content", "")
    else:
        poi = doc.copy()
    all_pois.append(poi)

print(f"POIデータ統一完了: {len(all_pois)}件")

# カテゴリ分布確認
from collections import Counter
categories = Counter(poi.get("category", "不明") for poi in all_pois)
print(f"カテゴリ数: {len(categories)}")

## Section 3: Phase 6モジュールのテスト

In [ ]:
# 3.1 geo_utilsのテスト
from src.geo_utils import (
    SHIBUYA_STATION,
    compute_spatial_info,
    enrich_all_pois
)

print(f"基準点: {SHIBUYA_STATION['name']}")
print(f"座標: ({SHIBUYA_STATION['lat']}, {SHIBUYA_STATION['lon']})")

# 最初のPOIで動作確認
if all_pois and all_pois[0].get("lat"):
    test_poi = all_pois[0]
    info = compute_spatial_info(test_poi["lat"], test_poi["lon"])
    print(f"\nテストPOI: {test_poi.get('name', '不明')}")
    print(f"  距離: {info.distance_m:.1f}m")
    print(f"  方向: {info.direction} ({info.direction_jp})")

In [ ]:
# 3.2 全POIに空間情報を追加
enriched_pois = enrich_all_pois(all_pois)
print(f"空間情報追加完了: {len(enriched_pois)}件")

# 方向別の分布確認
direction_counts = Counter(poi.get("direction_from_station", "不明") for poi in enriched_pois)
print("\n方向別分布:")
for direction, count in direction_counts.most_common():
    print(f"  {direction}: {count}件")

In [ ]:
# 3.3 aggregatorのテスト
from src.aggregator import (
    compare_east_west,
    compare_categories,
    get_top_categories,
    analyze_category_by_direction,
    filter_by_category
)

# 東西比較（全体）
ew_all = compare_east_west(enriched_pois)
print(f"【東西比較（全体）】")
print(f"  {ew_all.to_japanese()}")

# 東西比較（カフェ）
ew_cafe = compare_east_west(enriched_pois, "カフェ")
print(f"\n【東西比較（カフェ）】")
print(f"  {ew_cafe.to_japanese()}")

# カテゴリランキング
top_cats = get_top_categories(enriched_pois, top_n=5)
print(f"\n【カテゴリランキング TOP5】")
for i, cat in enumerate(top_cats, 1):
    print(f"  {i}. {cat.category}: {cat.count}件")

In [ ]:
# 3.4 質問分析のテスト
from src.structured_rag_system import analyze_question

test_questions = [
    "渋谷駅の東側と西側、どちらにカフェが多いですか？",
    "渋谷駅周辺の映画館と劇場、どちらが多いですか？",
    "渋谷駅周辺で最も多いPOIカテゴリは何ですか？",
    "渋谷駅から500m以内のコンビニを教えてください",
    "渋谷のカフェを教えてください"
]

print("【質問分析テスト】")
for q in test_questions:
    analysis = analyze_question(q)
    print(f"\n質問: {q}")
    print(f"  タイプ: {analysis.question_type}")
    print(f"  カテゴリ: {analysis.subcategories}")
    print(f"  方向: {analysis.directions}")
    print(f"  距離制約: {analysis.distance_constraint}")

## Section 4: モデルロード

In [ ]:
# 4.1 LLMモデルロード
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"Loading {LLM_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"LLMモデルロード完了")

In [ ]:
# 4.2 Embeddingモデルロード
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"Embeddingモデルロード完了")

## Section 5: ベクトルストア構築

In [ ]:
# 5.1 ベクトルストア構築
from langchain_chroma import Chroma
from langchain_core.documents import Document

documents = []
for poi in poi_documents:
    if "metadata" in poi:
        documents.append(Document(
            page_content=poi["content"],
            metadata=poi["metadata"]
        ))
    else:
        content = poi.get("content", f"{poi.get('name', '')} - {poi.get('category', '')}")
        documents.append(Document(
            page_content=content,
            metadata=poi
        ))

print(f"ベクトルストア構築中... ({len(documents)}件)")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="poi_shibuya_phase6"
)
print(f"ベクトルストア構築完了")

## Section 6: 構造化RAGシステム初期化

In [ ]:
# 6.1 構造化RAGシステム初期化
from src.structured_rag_system import StructuredRAGSystem

structured_rag = StructuredRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    all_pois=all_pois,
    debug=True
)
print("構造化RAGシステム初期化完了")

## Section 7: 構造化RAGテスト

In [ ]:
# 7.1 spatial_comparison改善テスト
spatial_test_questions = [
    "渋谷駅の東側と西側、どちらにカフェが多いですか？",
    "渋谷駅周辺の映画館と劇場、どちらが多いですか？それぞれの数も教えてください",
    "渋谷駅周辺で最も多いPOIカテゴリは何ですか？上位3つを教えてください"
]

print("=" * 70)
print("spatial_comparison改善テスト")
print("=" * 70)

for q in spatial_test_questions:
    print(f"\n質問: {q}")
    print("-" * 50)
    
    result = structured_rag.query_with_structured_rag(q)
    
    print(f"分析タイプ: {result['analysis']['question_type']}")
    print(f"処理時間: {result['time_ms']}ms")
    print(f"\n回答:\n{result['answer'][:800]}")
    print("=" * 70)

In [ ]:
# 7.2 RAGあり/なし比較
print("=" * 70)
print("RAGあり/なし比較")
print("=" * 70)

comparison_question = "渋谷駅の東側と西側、どちらにカフェが多いですか？"
comparison_result = structured_rag.compare(comparison_question)

## Section 8: Phase 5テストケースでの評価

In [ ]:
# 8.1 テストケース読み込み
try:
    from src.test_cases_v2 import TEST_CASES_V2, get_test_cases_by_subcategory
    print(f"テストケース読み込み完了: {len(TEST_CASES_V2)}件")
except ImportError:
    print("test_cases_v2.pyが見つかりません。手動でテストを実行してください。")
    TEST_CASES_V2 = None

In [ ]:
# 8.2 spatial_comparisonテストケースの抽出と評価
if TEST_CASES_V2:
    spatial_cases = [tc for tc in TEST_CASES_V2 if tc.subcategory == "spatial_comparison"]
    print(f"spatial_comparisonテストケース: {len(spatial_cases)}件")
    
    for tc in spatial_cases[:3]:  # 最初の3件のみテスト
        print(f"\n--- {tc.id}: {tc.description} ---")
        print(f"質問: {tc.prompt}")
        
        result = structured_rag.query_with_structured_rag(tc.prompt)
        print(f"回答:\n{result['answer'][:500]}...")

## Section 9: 結果保存

In [ ]:
# 9.1 結果保存
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 集計データの保存
aggregation_data = {
    "timestamp": timestamp,
    "phase": "Phase 6 - Structured RAG",
    "poi_count": len(enriched_pois),
    "direction_distribution": dict(direction_counts),
    "top_categories": [c.to_dict() for c in get_top_categories(enriched_pois, 10)],
    "east_west_comparison": compare_east_west(enriched_pois).to_dict()
}

output_path = f"{RESULTS_DIR}/phase6_aggregation_{timestamp}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(aggregation_data, f, ensure_ascii=False, indent=2)

print(f"結果保存完了: {output_path}")

## Section 10: サマリー

In [ ]:
# 10.1 Phase 6サマリー
print("=" * 70)
print("Phase 6: 構造化RAGシステム サマリー")
print("=" * 70)
print(f"\n【データ】")
print(f"  POI総数: {len(enriched_pois)}件")
print(f"  カテゴリ数: {len(categories)}種類")

print(f"\n【新機能】")
print("  ✅ 座標計算（距離・方向判定）")
print("  ✅ エリアクラスタリング")
print("  ✅ カテゴリ別集計")
print("  ✅ 方向別集計")
print("  ✅ 東西/南北比較")
print("  ✅ カテゴリ間比較")
print("  ✅ 質問分析・戦略選択")

print(f"\n【東西比較（全体）】")
print(f"  {compare_east_west(enriched_pois).to_japanese()}")

print(f"\n【次のステップ】")
print("  1. Phase 5テストケース（55件）での完全評価")
print("  2. spatial_comparison/advanced_comparisonの改善度測定")
print("  3. グラフRAGの実装（Phase 6.3）")